# Stage 4 — locate the noise threshold, and price the guarantee it carries

## The gap this closes

| noise multiplier | leakage | where measured |
|---|---|---|
| **0.000** | 45 verbatim hits, recitation **0.270** | §6.5.3 clipping control |
| **0.098** | 0 hits, recitation **0.000** | §6.5.2, the loosest budget swept |

Every budget an operator would consider sits above 0.098. **The entire
transition lies inside an interval no experiment has entered.** Locating it
gives the smallest perturbation that suffices — the quantity §6.5.4 argues an
operator actually needs, because within the swept range the choice of epsilon
changes the strength of the guarantee and nothing that can be measured.

## Why the answer is likely to be uncomfortable

At the 560 lots of this configuration the accountant prices the ladder like this:

| nm | 0.000 | 0.010 | 0.020 | 0.040 | 0.070 | 0.098 |
|---|---|---|---|---|---|---|
| **ε** | unbounded | **81.6** | **28.0** | **11.5** | **5.9** | 3.99 |

If the empirical threshold sits at a small multiplier, the guarantee attached to
the smallest sufficient noise is **weak to the point of vacuity** — an epsilon in
the tens or hundreds is not a privacy statement anyone would report.

That would *sharpen* §6.5.4, not soften it. The operator who chooses ε = 0.5 is
paying for guarantee strength; this experiment says how much of that payment
buys observable protection. The expected answer is none of it.

## Two instruments

**Recitation leads.** On the full matrix the signal is 45 hits in 11,760 probes
— a rate of 0.0038. On the held-out probe it is 0.270 against a base rate of
exactly zero. That is roughly seventy times stronger per query and ten times
cheaper to collect. A stratified quarter of the matrix confirms it; at nm = 0
that quarter should return about eleven hits, so returning zero would be a
one-in-fifty-thousand event. The sample is not the weak link.

## Two seeds, because §6.5.3 happened

§6.5.3 established that a single run at this operating point is a draw, not a
result — it is where the ε = 0.50 utility claim died. **Every rung here is run at
two seeds, and the readout refuses to quote a threshold the seeds disagree
about.** Running one seed and reading a boundary off it would be repeating the
exact mistake the dissertation now devotes a section to.

The **nm = 0.000 rung is not padding**: it re-runs the zero-noise control at a
fresh seed, so the ladder has to reproduce a known endpoint (45 hits, 0.270)
before its interior is believed.

---

**Before starting:** Runtime → Change runtime type → **L4 GPU**.
Seed 1 ≈ 4 h (cell 5), seed 2 ≈ 4 h (cell 7). Checkpointed per rung — run them
in separate sessions if you like.

## 1 — Confirm the GPU

In [ ]:
!nvidia-smi

## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/piibench'
os.makedirs(DRIVE, exist_ok=True)
print('results will be written to:', DRIVE)

## 3 — Unpack the bundle

Upload `piibench_threshold.zip` into the **`piibench` folder** of your Drive first, then run this cell.

In [ ]:
import zipfile, os
src = f'{DRIVE}/piibench_threshold.zip'
assert os.path.exists(src), f'not found: {src} - upload the zip to Drive first'
zipfile.ZipFile(src).extractall('/content/work')
os.chdir('/content/work')
print(sorted(os.listdir('.')))

## 4 — Install dependencies
(about 2 minutes)

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets pandas

## 5 — Seed 1: the first ladder (about 4 hours)

Five rungs: nm 0.000, 0.010, 0.020, 0.040, 0.070. Each is trained, then audited
on a stratified quarter of the matrix plus all 900 held-out probes.

The readout at the end will say **PROVISIONAL** — that is correct and expected.
One seed cannot settle a boundary. Read it to see roughly where the transition
is, then run cell 7.

In [ ]:
!python noise_threshold.py \
  --model Qwen/Qwen2.5-1.5B \
  --persons 140 --epochs 2 --repeats 2 \
  --heldout-persons 180 \
  --multipliers 0.0 0.010 0.020 0.040 0.070 \
  --runs 1 --matrix-fraction 0.25 \
  --out-dir $DRIVE/threshold_140 \
  --adapter-dir /content/adapters_thr

## 6 — Read the provisional ladder

In [ ]:
import json, math
r = json.load(open(f'{DRIVE}/threshold_140/threshold.json'))
rungs = {}
for k, v in r.items():
    rungs.setdefault(v['noise_multiplier'], {})[v['run']] = v
print(f"  {'nm':>7} {'eps':>11} {'seed':>5} {'hits':>10} {'recite':>9} {'novel':>8}  verdict")
for nm in sorted(rungs):
    for run in sorted(rungs[nm]):
        v = rungs[nm][run]
        e = 'unbounded' if v.get('epsilon') is None else f"{v['epsilon']:.2f}"
        leaks = v['recitation'] > 0.02 or v['raw_hits'] > 0
        print(f"  {nm:>7.3f} {e:>11} {run:>5} "
              f"{str(v['raw_hits'])+'/'+str(v['n_probes']):>10} "
              f"{v['recitation']:>9.4f} {v['novel_valid']:>8.4f}  {'LEAKS' if leaks else 'clean'}")
print(f"\n  {'0.000':>7} {'unbounded':>11} {'ref':>5} {'45/11760':>10} {0.27:>9.4f} {0.4078:>8.4f}  LEAKS  (6.5.3)")
print(f"  {'0.098':>7} {'3.99':>11} {'ref':>5} {'0/11760':>10} {0.0:>9.4f} {0.4144:>8.4f}  clean  (6.5.2)")
done = [nm for nm in rungs if len(rungs[nm]) >= 2]
print(f"\n  rungs with two seeds: {len(done)}/{len(rungs)}")
if len(done) < len(rungs):
    print('  PROVISIONAL - run cell 7 before quoting any threshold.')

## 7 — Seed 2: confirm it (about 4 hours)

Same five rungs, different training seed. This is the cell that turns a
provisional bracket into a reportable one.

**Can be a separate session.** Re-run cells 2 → 3 → 4 first; completed rungs are
detected and skipped.

In [ ]:
!python noise_threshold.py \
  --model Qwen/Qwen2.5-1.5B \
  --persons 140 --epochs 2 --repeats 2 \
  --heldout-persons 180 \
  --multipliers 0.0 0.010 0.020 0.040 0.070 \
  --runs 2 --matrix-fraction 0.25 \
  --out-dir $DRIVE/threshold_140 \
  --adapter-dir /content/adapters_thr

## 8 — Read the bracketed threshold

This is the cell whose output goes into the paper.

In [ ]:
import json, math
r = json.load(open(f'{DRIVE}/threshold_140/threshold.json'))
rungs = {}
for k, v in r.items():
    rungs.setdefault(v['noise_multiplier'], {})[v['run']] = v
def leaks(v): return v['recitation'] > 0.02 or v['raw_hits'] > 0
disagree = [nm for nm in rungs if len(rungs[nm]) >= 2
            and len({leaks(x) for x in rungs[nm].values()}) > 1]
complete = [nm for nm in rungs if len(rungs[nm]) >= 2 and nm not in disagree]
if disagree:
    print(f'  SEEDS DISAGREE at nm {sorted(disagree)} - those rungs are draws, not')
    print('  boundaries. Add seeds there before quoting anything.')
leaky = [nm for nm in complete if any(leaks(x) for x in rungs[nm].values())]
clean = [nm for nm in complete if nm not in leaky]
if leaky and clean:
    lo, hi = max(leaky), min(clean)
    e = rungs[hi][min(rungs[hi])].get('epsilon')
    print(f'  THRESHOLD BRACKETED: leaks at nm {lo:.3f}, clean by nm {hi:.3f}, both seeds.')
    print(f'  Smallest sufficient noise carries epsilon {e} at delta 1e-5.')
    if e and e > 10:
        print()
        print('  That is not a reportable privacy guarantee. So the protection an')
        print('  operator can actually observe is bought at a budget nobody would')
        print('  publish, and every tighter budget buys guarantee strength alone.')
        print('  This is the Section 6.5.4 argument with a number attached.')
elif complete and not leaky:
    print('  Every rung clean - transition is below this ladder. Check the nm=0.000')
    print('  rung reproduced the control (45 hits / 0.270); if not, suspect the run.')
elif complete and not clean:
    print('  Every rung leaks - transition is above this ladder, between the top')
    print('  rung and nm 0.098. Extend upward.')

## 9 — Collect the results

In [ ]:
import shutil, os, glob
os.makedirs('/content/send', exist_ok=True)
src = f'{DRIVE}/threshold_140'
for pat in ['threshold.json', 'dataset.json', 'records_*.csv']:
    for f in glob.glob(f'{src}/{pat}'):
        shutil.copy(f, '/content/send/')
sent = sorted(os.listdir('/content/send'))
for f in sent:
    print(f"  {f:<40} {os.path.getsize('/content/send/'+f)/1024:>9.1f} KB")
shutil.make_archive(f'{DRIVE}/threshold_results', 'zip', '/content/send')
print(f"\n  {len(sent)} files -> {DRIVE}/threshold_results.zip "
      f"({os.path.getsize(f'{DRIVE}/threshold_results.zip')/1e6:.1f} MB)")
assert any(f.startswith('records_matrix') for f in sent), 'per-query records missing'
print('  Download from Drive and send it back.')

---
### If the session drops
Re-run cells 2 → 3 → 4, then the cell you were on. Completed rungs are skipped and saved adapters reused when still on disk.

### If you hit CUDA out of memory
Add `--batch-size 12`.

### The one result that would invalidate the run
If the **nm = 0.000** rung comes back clean, the ladder has failed to reproduce the §6.5.3 control (45 hits, recitation 0.270) and nothing above it can be trusted. Check that rung first, before reading the interior of the ladder.

### If the transition is not inside the ladder
Everything leaking means it sits between the top rung and 0.098 — rerun with `--multipliers 0.080 0.090`. Everything clean above nm 0 means it sits below 0.010 — rerun with `--multipliers 0.002 0.005`.